# Building Research Agents with LangChain + LangGraph

**End goal:** a NotebookLM-style research workflow (plan → search → read → evaluate → find gaps → repeat → cite → report).

This notebook builds the **3 foundation agents** that everything else depends on, then wires them together with LangGraph.

| # | Agent | Why build it first | Core concept it teaches |
|---|---|---|---|
| 1 | **Planner** | Entry point: every other agent works off its questions | LLM call + **structured output** (a *chain*, not a loop) |
| 2 | **Paper Finder** | Your first real *agent*: it decides what to search and retries when results are bad | **Tools** + the **agent loop** (`create_agent`) |
| 3 | **Paper Reader / JEV** | The token-saving gate: score cheaply, only read what matters | **LangGraph** `StateGraph`, **conditional edges**, cheap vs. expensive models |
| ★ | **Manager** (the graph itself) | Connects 1 → 2 → 3 in parallel | **Fan-out** with `Send`, state **reducers** |
| ★ | **Chat memory** | Your "Chat Application" requirement: separate memory per chat | **Checkpointers** + `thread_id` |

Gaps, New Questions, Fact Check, Citation and Reporter reuse these same patterns (see **Part 6** at the end).

### Mental model
- **LangChain** = building blocks: chat models, messages, tools, structured output, `create_agent`.
- **LangGraph** = orchestration: shared **state**, **nodes** (functions), **edges** (what runs next), loops, parallelism, memory.
- `create_agent` from LangChain is itself a small LangGraph graph under the hood (model node ⇄ tools node).

## 0. Setup

Install once from the repo root (this is a `uv` project), then restart the kernel:

```bash
uv add langchain langchain-openai langgraph
```

The OpenAI key is read from `.env` (`OPENAI_API_KEY`), which is already gitignored.

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from .env into the environment
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - check your .env file"

from langchain.chat_models import init_chat_model

# Two models: a capable one for real work, a cheap one for high-volume scoring (JEV).
llm = init_chat_model("openai:gpt-5-mini")
scorer_llm = init_chat_model("openai:gpt-6-luna")

### Warm-up: talking to a model

Every chat model takes a **list of messages** and returns an `AIMessage`.
- `SystemMessage` = instructions / role for the model
- `HumanMessage` = the user's input
- `AIMessage` = the model's reply (may also contain *tool calls*, see Part 2)

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage

response = llm.invoke([
    SystemMessage("You are a concise tutor. Answer in one sentence."),
    HumanMessage("What is the difference between LangChain and LangGraph?"),
])
print(type(response).__name__)
print(response.content)

AIMessage
LangChain is an open-source, code-first library for building LLM applications (chains, agents, prompts, memory, connectors) in Python/JS, whereas LangGraph is a visual/graph-based, low-code/hosted platform for designing, orchestrating, deploying, and managing LLM workflows with UI, collaboration, and observability features.


---
## Part 1: Planner Agent (structured output)

The planner turns a topic into research questions, one per category:

- **who**: key researchers in the field
- **major_works**: landmark books/papers/media (e.g. Said's *Orientalism*)
- **what**: current conversations and debates, and *when* they emerged
- **theories**: major theories related to the topic
- **applications**: how those theories are applied

**Key idea: structured output.** Free text is hard for the next agent to use. Instead we define a **Pydantic schema** and call `llm.with_structured_output(Schema)`, so the model *must* return an object that fits it. Every agent-to-agent handoff in this project should work like this.

> The planner is not really an "agent", because it doesn't loop or use tools. It's one well-prompted LLM call. **Only use an agent loop when the model has to make decisions over several steps.** That's cheaper and more predictable.

In [6]:
from typing import Literal
from pydantic import BaseModel, Field

Category = Literal["who", "major_works", "what", "theories", "applications"]


class ResearchQuestion(BaseModel):
    category: Category
    question: str = Field(description="A specific research question")
    search_query: str = Field(description="3-6 keyword query for an academic search engine")


class ResearchPlan(BaseModel):
    topic: str
    mode: Literal["academic", "practical"]
    questions: list[ResearchQuestion]


PLANNER_PROMPT = """You are a research planner. Create a background research plan for the user's topic.

Mode: {mode}
- academic: the user is starting background research on a new academic topic.
- practical: the user wants to learn a new technology or idea well enough to use it.

Write exactly one question for each category:
- who: key researchers or practitioners and their groups
- major_works: landmark books, papers, or media (e.g. Edward Said's Orientalism for post-colonial studies)
- what: current conversations, debates and disagreements, and roughly when they emerged
- theories: major theories or frameworks related to the topic
- applications: how those theories are applied in practice

For each question also write a short keyword search_query suitable for an academic search engine."""

planner = llm.with_structured_output(ResearchPlan)


def run_planner(topic: str, mode: str = "academic") -> ResearchPlan:
    return planner.invoke([
        SystemMessage(PLANNER_PROMPT.format(mode=mode)),
        HumanMessage(topic),
    ])

def run_planner_practical(topic: str, mode: str = "practical") -> ResearchPlan:
    return planner.invoke([
        SystemMessage(PLANNER_PROMPT.format(mode=mode)),
        HumanMessage(topic),
    ])

In [7]:
plan = run_planner("retrieval-augmented generation for question answering")

print(type(plan))  # a real Pydantic object, not a string
for q in plan.questions:
    print(f"[{q.category}] {q.question}\n    query: {q.search_query}")

plan_practical = run_planner_practical("retrieval-augmented generation for question answering")
print(type(plan_practical))  # a real Pydantic object, not a string
for q in plan_practical.questions:
    print(f"[{q.category}] {q.question}\n    query: {q.search_query}")

<class '__main__.ResearchPlan'>
[who] Who are the leading researchers and research groups working on retrieval-augmented generation (RAG) for QA, and which labs, teams, or open-source projects should I follow?
    query: RAG researchers Facebook AI DeepMind OpenAI FAIR
[major_works] What are the landmark papers, datasets, toolkits, and blog posts that define RAG for QA (e.g., original RAG paper, REALM, DPR, evaluation suites) and which should be read first?
    query: Retrieval-Augmented Generation Lewis 2020 REALM DPR RAG
[what] What are the current research conversations, open debates, and major criticisms about RAG in QA (e.g., hallucination, retrieval quality, latency, evaluation metrics), and when did these debates emerge?
    query: RAG debates hallucination factuality retrieval evaluation 2020
[theories] What are the principal theoretical frameworks and architectures underlying RAG (e.g., dense vs. sparse retrieval, retrieval-then-generate vs. fusion-in-decoder, retrieval as lat

**Try it:** run with `mode="practical"` and compare. For the practical path you'll probably want different categories later (core concepts, how-to, tooling, pitfalls). You would add a second schema/prompt and choose between them by mode.

---
## Part 2: Paper Finder Agent (tools + the agent loop)

### 2a. Tools
A **tool** is just a Python function wrapped with `@tool`. LangChain turns the **function name, type hints and docstring** into a schema that the model reads. **The docstring is the prompt for the tool**, so write it for the LLM.

We'll search **arXiv** (free, no API key). arXiv is mostly STEM. For humanities topics (e.g. post-colonial studies) you would later swap in or add another source, such as OpenAlex, Semantic Scholar, or a general web-search tool. The agent code stays the same and only the tool changes.

In [8]:
import httpx
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

ATOM = {"a": "http://www.w3.org/2005/Atom"}


@tool
def search_arxiv(query: str, max_results: int = 5) -> list[dict]:
    """Search arXiv for academic papers matching ALL keywords in `query`.

    Returns a list of papers with title, authors, year, abstract and url.
    Use 2-5 keywords, e.g. 'retrieval augmented generation evaluation'.
    If nothing comes back, retry with fewer or broader keywords.
    """
    search = " AND ".join(f"all:{word}" for word in query.split())
    response = httpx.get(
        "https://export.arxiv.org/api/query",
        params={"search_query": search, "max_results": max_results, "sortBy": "relevance"},
        timeout=30,
        follow_redirects=True,
    )
    response.raise_for_status()

    papers = []
    for entry in ET.fromstring(response.text).findall("a:entry", ATOM):
        papers.append({
            "title": " ".join(entry.findtext("a:title", "", ATOM).split()),
            "authors": [a.findtext("a:name", "", ATOM) for a in entry.findall("a:author", ATOM)],
            "year": entry.findtext("a:published", "", ATOM)[:4],
            "abstract": " ".join(entry.findtext("a:summary", "", ATOM).split()),
            "url": entry.findtext("a:id", "", ATOM),
        })
    return papers


# Tools can be called directly - always test them on their own first.
for p in search_arxiv.invoke({"query": "retrieval augmented generation", "max_results": 3}):
    print(p["year"], p["title"], p["url"])

2025 AR-RAG: Autoregressive Retrieval Augmentation for Image Generation http://arxiv.org/abs/2506.06962v3
2024 Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation http://arxiv.org/abs/2411.18583v1
2024 EVOR: Evolving Retrieval for Code Generation http://arxiv.org/abs/2402.12317v2


In [9]:
# What the model "sees" about the tool:
print(search_arxiv.name)
print(search_arxiv.args)

# Binding a tool lets the model REQUEST a call. The model never runs anything itself.
msg = llm.bind_tools([search_arxiv]).invoke("Find papers about RAG hallucination")
print(msg.tool_calls)  # the model's request: which tool, which args

search_arxiv
{'query': {'title': 'Query', 'type': 'string'}, 'max_results': {'default': 5, 'title': 'Max Results', 'type': 'integer'}}
[{'name': 'search_arxiv', 'args': {'query': 'retrieval augmented generation hallucination', 'max_results': 8}, 'id': 'call_YX8TOxzie7bPbg1c6P2KzU0x', 'type': 'tool_call'}]


### 2b. The agent loop

Something has to *execute* those tool calls and send the results back to the model until it's done. That is the **agent loop** (also called ReAct):

```
        ┌──────────────────────────────┐
        ▼                              │
  [model] ── wants a tool? ── yes ──▶ [tools]
        │
        no
        ▼
     final answer (optionally structured via response_format)
```

`create_agent` builds this loop for you. With `response_format=Schema`, the final answer comes back as a Pydantic object in `result["structured_response"]`.

In [13]:
from langchain.agents import create_agent


class FoundPaper(BaseModel):
    title: str
    authors: list[str]
    year: str
    url: str
    abstract: str
    why_relevant: str = Field(description="One sentence: how this paper helps answer the question")


class PaperList(BaseModel):
    papers: list[FoundPaper]


PAPERS_PER_QUESTION = 3

FINDER_PROMPT = f"""You are a research paper finder. Given a research question, use the search_arxiv tool
to find the {PAPERS_PER_QUESTION} most relevant papers.

- Start with the suggested query. If results are empty or off-topic, search again with fewer or
  different keywords. Do at most 3 searches.
- Only return papers that actually appeared in the search results. Never invent papers.
- Copy title, authors, year, url and abstract exactly from the tool output.
- For each paper, explain in one sentence why it helps answer the question."""

finder_agent = create_agent(
    model=llm,
    tools=[search_arxiv],
    system_prompt=FINDER_PROMPT,
    response_format=PaperList,
)


def run_finder(question: ResearchQuestion) -> list[FoundPaper]:
    result = finder_agent.invoke({"messages": [{
        "role": "user",
        "content": f"Research question: {question.question}\nSuggested query: {question.search_query}",
    }]})
    return result["structured_response"].papers

In [14]:
# Run the finder on one question and inspect every step of the loop.
question = plan.questions[0]
result = finder_agent.invoke({"messages": [{
    "role": "user",
    "content": f"Research question: {question.question}\nSuggested query: {question.search_query}",
}]})

for m in result["messages"]:
    m.pretty_print()  # Human -> AI (tool call) -> Tool (results) -> AI ... repeat

found_papers = result["structured_response"].papers
print("\n=== Structured result ===")
for p in found_papers:
    print(f"- {p.title} ({p.year})\n  {p.why_relevant}")

ReadTimeout: The read operation timed out

---
## Part 3: Paper Reader / JEV Agent (your first LangGraph graph)

Goal: **don't spend tokens reading bad papers.**
1. **Score** (cheap model): read only title + abstract and give a JEV relevance score from 1 to 10.
2. **Branch**: if score ≥ threshold, **read** (capable model extracts thesis, key points, evidence, limitations). Otherwise **skip**.

This is a perfect first graph because it has a **decision point**. In LangGraph terms:
- **State**: a `TypedDict` shared by all nodes. Each node gets the state and returns a **partial update** (a dict with only the keys it changed).
- **Nodes**: plain Python functions.
- **Edges**: fixed transitions (`add_edge`) or **conditional** ones (`add_conditional_edges`), where a function looks at the state and returns the name of the next node.

```
START → score_paper ──(score ≥ 6)──▶ read_paper → END
                    └─(score < 6)──▶ skip_paper → END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

JEV_THRESHOLD = 6


class RelevanceScore(BaseModel):
    score: int = Field(ge=1, le=10)
    reasoning: str = Field(description="One sentence justifying the score")


class PaperNotes(BaseModel):
    thesis: str = Field(description="The paper's main claim in one sentence")
    key_points: list[str] = Field(description="3-5 points relevant to the research question")
    evidence_type: str = Field(description="e.g. experiment, survey, case study, theoretical")
    limitations: list[str]


class ReaderState(TypedDict):
    question: str      # input
    paper: dict        # input
    score: int         # set by score_paper
    reasoning: str     # set by score_paper
    notes: dict | None # set by read_paper / skip_paper
    status: str        # "read" or "skipped"


SCORE_PROMPT = """Rate how useful this paper is for answering the research question, from 1 to 10.
9-10: directly answers it or is a landmark work. 6-8: useful supporting source.
3-5: tangential. 1-2: irrelevant.

Research question: {question}
Title: {title}
Abstract: {abstract}"""

READ_PROMPT = """Extract research notes from this paper that help answer the research question.
Only use what the text says. Do not add outside knowledge.

Research question: {question}
Title: {title}
Text: {text}"""

scorer = scorer_llm.with_structured_output(RelevanceScore)
extractor = llm.with_structured_output(PaperNotes)


def score_paper(state: ReaderState) -> dict:
    paper = state["paper"]
    result = scorer.invoke(SCORE_PROMPT.format(
        question=state["question"], title=paper["title"], abstract=paper["abstract"][:1500]
    ))
    return {"score": result.score, "reasoning": result.reasoning}


def route_after_score(state: ReaderState) -> Literal["read_paper", "skip_paper"]:
    return "read_paper" if state["score"] >= JEV_THRESHOLD else "skip_paper"


def read_paper(state: ReaderState) -> dict:
    paper = state["paper"]
    # TODO: upgrade from the abstract to real text (first page + conclusion),
    # e.g. download the PDF from the arXiv url and extract it with pypdf.
    notes = extractor.invoke(READ_PROMPT.format(
        question=state["question"], title=paper["title"], text=paper["abstract"]
    ))
    return {"notes": notes.model_dump(), "status": "read"}


def skip_paper(state: ReaderState) -> dict:
    return {"notes": None, "status": "skipped"}


builder = StateGraph(ReaderState)
builder.add_node("score_paper", score_paper)
builder.add_node("read_paper", read_paper)
builder.add_node("skip_paper", skip_paper)
builder.add_edge(START, "score_paper")
builder.add_conditional_edges("score_paper", route_after_score)
builder.add_edge("read_paper", END)
builder.add_edge("skip_paper", END)

reader_graph = builder.compile()

In [ ]:
from IPython.display import Image, Markdown, display


def show_graph(graph):
    """Render a compiled graph. Falls back to Mermaid text if the renderer is unreachable."""
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())


show_graph(reader_graph)

In [ ]:
# Run the reader on the papers the finder returned in Part 2.
for p in found_papers:
    out = reader_graph.invoke({"question": question.question, "paper": p.model_dump()})
    print(f"[JEV {out['score']}/10 - {out['status']}] {p.title}\n    {out['reasoning']}")
    if out["notes"]:
        print(f"    thesis: {out['notes']['thesis']}")

---
## Part 4: The Manager, a LangGraph workflow that connects all three

Your "Manager Agent" doesn't need to be an LLM. **The graph *is* the manager.** When the order of steps is known (plan → find → read), fixed edges are cheaper, faster and more reliable than asking an LLM what to do next. Save LLM-based routing for decisions that really change from run to run, such as the Gaps agent deciding whether to dig deeper.

Two new LangGraph concepts:

1. **`Send` (fan-out / map-reduce):** a conditional edge can return a *list* of `Send("node", payload)`, one per item. LangGraph then runs that node **in parallel**, once per payload. We use it to run one finder per question and one reader per paper.
2. **Reducers:** when parallel nodes all write to the same key, LangGraph needs to know how to merge them. `Annotated[list, operator.add]` means "append" instead of "overwrite".

```
START → plan ─Send×5─▶ find_papers ─▶ collect ─Send×N─▶ read_paper ─▶ jev_report → END
              (one per question)   (dedupe)   (one per paper)
```

In [ ]:
import operator
from typing import Annotated
from langgraph.types import Send


class ResearchState(TypedDict):
    topic: str
    mode: str
    plan: ResearchPlan
    found: Annotated[list[dict], operator.add]      # parallel finders append here
    to_read: list[dict]                             # deduplicated papers
    evaluated: Annotated[list[dict], operator.add]  # parallel readers append here
    jev_report: str


# Payload types for the parallel workers (what each Send carries)
class FinderTask(TypedDict):
    research_question: ResearchQuestion


class ReaderTask(TypedDict):
    question_text: str
    paper: dict


def plan_node(state: ResearchState) -> dict:
    return {"plan": run_planner(state["topic"], state["mode"])}


def dispatch_finders(state: ResearchState) -> list[Send]:
    return [Send("find_papers", {"research_question": q}) for q in state["plan"].questions]


def find_papers_node(task: FinderTask) -> dict:
    q = task["research_question"]
    papers = run_finder(q)
    return {"found": [{"question": q.question, "paper": p.model_dump()} for p in papers]}


def collect_node(state: ResearchState) -> dict:
    seen, unique = set(), []
    for item in state["found"]:
        url = item["paper"]["url"]
        if url not in seen:
            seen.add(url)
            unique.append(item)
    return {"to_read": unique}


def dispatch_readers(state: ResearchState):
    if not state["to_read"]:
        return "jev_report"
    return [
        Send("read_paper", {"question_text": item["question"], "paper": item["paper"]})
        for item in state["to_read"]
    ]


def read_paper_node(task: ReaderTask) -> dict:
    # A compiled graph can be called from inside another graph's node (a subgraph).
    result = reader_graph.invoke({"question": task["question_text"], "paper": task["paper"]})
    return {"evaluated": [result]}


def jev_report_node(state: ResearchState) -> dict:
    rows = sorted(state.get("evaluated", []), key=lambda r: r["score"], reverse=True)
    lines = ["| JEV | Status | Paper | Reasoning |", "|---|---|---|---|"]
    for r in rows:
        p = r["paper"]
        reasoning = r["reasoning"].replace("|", "/")
        lines.append(f"| {r['score']}/10 | {r['status']} | [{p['title']}]({p['url']}) ({p['year']}) | {reasoning} |")
    return {"jev_report": "\n".join(lines)}


builder = StateGraph(ResearchState)
builder.add_node("plan", plan_node)
builder.add_node("find_papers", find_papers_node)
builder.add_node("collect", collect_node)
builder.add_node("read_paper", read_paper_node)
builder.add_node("jev_report", jev_report_node)

builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", dispatch_finders, ["find_papers"])
builder.add_edge("find_papers", "collect")  # waits for ALL parallel finders to finish
builder.add_conditional_edges("collect", dispatch_readers, ["read_paper", "jev_report"])
builder.add_edge("read_paper", "jev_report")
builder.add_edge("jev_report", END)

research_graph = builder.compile()
show_graph(research_graph)

### Run it
`stream()` shows progress while the graph runs. `stream_mode="updates"` yields each node's output as it finishes, and `"values"` yields the full state after each step (the last one is the final state).

⚠️ This makes about 5 finder runs plus up to 15 scoring calls and a few reads. With the mini/nano models that's cheap, but not free.

In [ ]:
inputs = {"topic": "retrieval-augmented generation for question answering", "mode": "academic"}

final_state = {}
for mode, chunk in research_graph.stream(inputs, stream_mode=["updates", "values"]):
    if mode == "updates":
        print("finished:", ", ".join(chunk.keys()))
    else:
        final_state = chunk

read = [r for r in final_state["evaluated"] if r["status"] == "read"]
print(f"\nFound {len(final_state['found'])} papers, {len(final_state['to_read'])} unique, {len(read)} passed JEV")

In [ ]:
# Output: human-readable JEV ratings for every paper (clickable links)
display(Markdown(final_state["jev_report"]))

In [ ]:
# The saved notes: this is what the future Reporter / Citation agents will consume
for r in read:
    print(f"## {r['paper']['title']}\nQ: {r['question']}")
    print(f"Thesis: {r['notes']['thesis']}")
    for point in r["notes"]["key_points"]:
        print(f"  - {point}")
    print()

---
## Part 5: Chat memory (separate context per chat)

For the Chat Application, each chat needs its own memory. In LangGraph that is a **checkpointer**: it saves the graph state after every step, keyed by a **`thread_id`**.
- Same `thread_id` → the conversation continues with full history.
- New `thread_id` → a fresh chat with no memory.

`InMemorySaver` is lost when the kernel restarts. For a real app, use `SqliteSaver` (`uv add langgraph-checkpoint-sqlite`) or Postgres. The code stays the same apart from the saver. The same `checkpointer=` argument also works on `research_graph` (`builder.compile(checkpointer=...)`), so each research project can be its own thread.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

chat_agent = create_agent(
    model=llm,
    tools=[search_arxiv],
    system_prompt="You are a helpful research assistant. Use search_arxiv when the user asks about papers.",
    checkpointer=InMemorySaver(),
)


def chat(message: str, thread_id: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = chat_agent.invoke({"messages": [{"role": "user", "content": message}]}, config)
    return result["messages"][-1].content


print("chat-1:", chat("Hi! I'm researching post-colonial literature.", "chat-1"))
print("chat-1:", chat("What topic am I researching?", "chat-1"))  # remembers
print("chat-2:", chat("What topic am I researching?", "chat-2"))  # new chat, no memory

In [ ]:
# You can inspect any chat's saved state - useful for a "chat history" sidebar later.
snapshot = chat_agent.get_state({"configurable": {"thread_id": "chat-1"}})
for m in snapshot.values["messages"]:
    print(f"{m.type}: {str(m.content)[:80]}")

---
## Part 6: Next steps, building the remaining agents with these patterns

| Agent | Pattern to reuse | Notes |
|---|---|---|
| **Identify Gaps** | structured output (Part 1) + conditional edge that **loops back** | Returns `{gaps: [...], done: bool}` |
| **New Research Questions** | same as Planner, but its input is the gaps | Output is `list[ResearchQuestion]`, so it plugs straight into `dispatch_finders` |
| **Fact Check** | `create_agent` with its own search tool (Part 2) + `reader_graph` as a subgraph (Part 3) | Schema: `{claim, supporting: [...], contradicting: [...], verdict: supported/contested/unsupported/no_evidence}`. This shows both sides instead of hiding "bad" evidence, and "no evidence" is a valid result. |
| **Citation** | fan-out with `Send` (Part 4) | Tip: format citations with **plain Python** from the metadata (authors, year, title, url). It's deterministic and costs nothing. Use an LLM only for messy metadata. |
| **Works Cited** | a plain node | Sort and join the citations into Markdown with clickable links |
| **Reporter** | a single LLM call over the saved notes | Give each paper an ID like `[3]` and tell the model to cite only those IDs, so every claim traces back to a source |

### Your gaps-depth question: a better way than "never more than 2 subquestions down"
Keep the depth limit, but enforce it **in the state**, not in the prompt, and add two more guards:

```python
class ResearchState(TypedDict):
    ...
    depth: int                                        # how many gap rounds so far
    explored_queries: Annotated[list[str], operator.add]  # never search the same thing twice
    gaps: list[str]

MAX_DEPTH = 2
MAX_PAPERS = 40  # overall budget

def after_gaps(state) -> Literal["new_questions", "synthesize"]:
    if (not state["gaps"]
            or state["depth"] >= MAX_DEPTH
            or len(state["evaluated"]) >= MAX_PAPERS):
        return "synthesize"
    return "new_questions"

builder.add_conditional_edges("identify_gaps", after_gaps)
builder.add_edge("new_questions", ...)  # -> dispatch_finders again, depth + 1
```

1. **Hard stop in code** (`depth`, `MAX_PAPERS`): an LLM can't talk its way past it.
2. **Relevance to the *original* topic**: have the gaps agent score each gap against `state["topic"]` (the same JEV idea) and keep only high scorers. This stops drift better than depth alone.
3. **Dedup** against `explored_queries` so loops don't re-search the same ground.

LangGraph also has a built-in safety net: `graph.invoke(..., {"recursion_limit": 50})`.

### When this notebook gets big
Move each piece into a module and keep the notebook for experiments:
```
research_agent/
    schemas.py      # all Pydantic models + state TypedDicts
    tools.py        # search_arxiv, web search, pdf reader
    agents/         # planner.py, finder.py, reader.py, gaps.py, ...
    graph.py        # builds research_graph
    chat.py         # chat app + checkpointer
```